# HanziGen - 字型生成训练（ModelScope DSW 整合版）

## 使用前准备

1. **打开 ModelScope DSW Notebook** ，选择 GPU 实例（推荐 V100）或 CPU 实例（数据准备阶段）
2. **上传本 notebook** 至 ModelScope Workspace（持久化目录）
3. **上传目标字体**（`.ttf` 或 `.otf`，文件名用英文）到 ModelScope Workspace 任意位置
4. （可选）**上传 Jigmo 参考字体**（`jigmo.ttf`、`jigmo2.ttf`、`jigmo3.ttf`）到 ModelScope Workspace
5. 按下方三阶段硬件策略选择实例，从 Cell 0 开始按顺序运行

---

## 关于持久化与临时目录

| 路径 | 说明 |
|---|---|
| **ModelScope Workspace** | 持久化存储，文件长期保留，重启不丢失 |
| **DSW-CPU /mnt/workspace/** | 临时工作目录，释放实例后可能清除 |

**建议**：本 notebook + 项目代码 + checkpoints 放在 ModelScope Workspace。Cell 1 会自动定位。

---

## 三阶段硬件策略

| 阶段 | 内容 | 推荐实例 | 理由 |
|---|---|---|---|
| 一、数据准备 | Cell 0-2：覆盖率分析、渲染字形图、划分字集 | CPU 实例 | 纯文件管理与字体渲染 |
| 二、模型训练 | Cell 0, 1, 3, 4：VQ-VAE + LDM | GPU：V100/A10 首选；T4 省钱但慢 | 训练需高频读写海量潜在特征 |
| 三A、推理+指标 | Cell 0, 1, 5：生成缺失字形 PNG | GPU：T4 即可 | 推理为批量计算 |
| 三B、导出+下载 | Cell 0, 6-8：SVG 向量化、打包下载 | CPU：多核大内存 首选 | 1万+ 小文件写入，瓶颈在内存缓存 |
| Checkpoint迁移 | Cell CK1/CK2：跨平台导出/导入 | 任意实例 | 打包或解压 checkpoint |

每次切换实例后，务必重跑 Cell 0 + Cell 1

---

## 流程总览

```
Cell 0:     配置字体名 + 阶段开关 + 补字基准 + 性能档位         任意实例
Cell 1:     环境初始化 + 硬件调优 + 字体/Jigmo 检测             任意实例，可重复运行
Cell 2:     数据准备（分析 -> 渲染 -> 划分字集）                  CPU 实例即可
Cell 3:     训练 VQ-VAE（自动精确续训）                         GPU 实例
Cell 4前置: 离线放置 VGG16 权重（手动上传 pth 时运行）          任意实例
Cell 4:     训练 LDM（自动精确续训）                            GPU 实例
Cell 5:     推理生成 + 评估指标                                 GPU 实例
Cell 6:     SVG 向量化 + 产出总览                               CPU 多核实例
Cell 7:     打包下载 SVG zip                                    CPU 多核实例
Cell 8:     打包下载全部产出（PNG + SVG）                        CPU 多核实例
Cell CK1:   Checkpoint 导出（跨平台迁移源端）                    任意实例
Cell CK2:   Checkpoint 导入（跨平台迁移目标端）                  任意实例
```

---
## Cell 0: 配置参数

只改这里！填你的字体文件名，支持自动搜索整个工作空间。

In [ ]:
# ==================== 修改你的字体文件名 ====================
TARGET_FONT = "your_font.otf"
# ==========================================================

FONT_NAME = TARGET_FONT.rsplit(".", 1)[0]

# ==================== 训练阶段开关（默认全开）====================
DO_DATA_PREP   = True     # Cell 2: 数据准备（CPU 实例即可）
DO_TRAIN_VQVAE = True     # Cell 3: 训练 VQ-VAE（GPU 实例）
DO_TRAIN_LDM   = True     # Cell 4: 训练 LDM（GPU 实例）
DO_INFERENCE   = True     # Cell 5: 推理 + 指标（GPU 实例）
# ==========================================================

# ==================== 补字基准（Cell 5 推理阶段生效）====================
CHARSET_BASE = "jf7000"  # jf7000 / unihan / gbk / gb2312
# ==========================================================

# ==================== 硬件性能档位 ====================
HW_PROFILE = "auto"  # auto / v100 / a10 / t4 / cpu
# ==========================================================

STATE_FILE = "colab_state.json"

print(f"目标字体: {TARGET_FONT}")
print(f"字体名称: {FONT_NAME}")
print(f"阶段开关: 数据准备={DO_DATA_PREP} VQVAE={DO_TRAIN_VQVAE} LDM={DO_TRAIN_LDM} 推理={DO_INFERENCE}")
print(f"补字基准: {CHARSET_BASE} | 性能档位: {HW_PROFILE}")

---
## Cell 1: 全自动环境初始化 + 硬件调优 + 断连自检

克隆仓库 -> 安装依赖 -> 智能搜索字体与 Jigmo -> 改写脚本 -> 字体切换检测 -> GPU 软检测 + 参数自动调优

专为 ModelScope DSW 设计：自动递归扫描所有工作区根目录，定位字体文件和已上传的 jigmo 文件。

可在任何实例重复运行；无 GPU 不会报错。

In [ ]:
import os, shutil, json, re, sys, glob, zipfile, io, urllib.request, subprocess
from pathlib import Path
from fontTools.ttLib import TTFont

# ===== 0. 发现所有可能的持久化和临时工作目录 =====
def discover_workspace_roots():
    roots = []
    candidates = [os.getcwd(), "/mnt/workspace", "/workspace", "/home", "/root"]
    seen = set()
    for c in candidates:
        if os.path.isdir(c) and c not in seen:
            seen.add(os.path.realpath(c))
            roots.append(c)
    return roots

WORKSPACE_ROOTS = discover_workspace_roots()
print(f"[目录] 检测到 {len(WORKSPACE_ROOTS)} 个工作目录:")
for r in WORKSPACE_ROOTS:
    print(f"  - {r}")

# ===== 1. 克隆项目代码 =====
REPO_URL  = "https://github.com/ICW-k/HanziGen_ICWfork.git"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")

is_project_root = os.path.isdir("scripts") and os.path.exists("requirements.txt")

if not is_project_root:
    found_repo = None
    search_paths = WORKSPACE_ROOTS + ["/"]
    for sr in search_paths:
        if not os.path.isdir(sr): continue
        try:
            for entry in sorted(os.listdir(sr)):
                candidate = os.path.join(sr, entry)
                if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, "scripts")):
                    if os.path.exists(os.path.join(candidate, "requirements.txt")):
                        found_repo = candidate
                        break
        except PermissionError: continue
        if found_repo: break

    if found_repo:
        print(f"\n发现已有项目目录: {found_repo}")
        os.chdir(found_repo)
    else:
        clone_target = None
        for r in WORKSPACE_ROOTS:
            potential = os.path.join(r, REPO_NAME)
            if os.path.isdir(potential):
                clone_target = potential
                break
        if not clone_target:
            clone_target = os.path.abspath(os.path.join(WORKSPACE_ROOTS[0] if WORKSPACE_ROOTS else ".", REPO_NAME))
        print(f"\n正在克隆仓库: {REPO_URL} -> {clone_target}")
        parent_dir = os.path.dirname(clone_target)
        os.makedirs(parent_dir, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, clone_target], check=True)
        os.chdir(clone_target)
    print(f"已切换到项目根目录: {os.getcwd()}")
else:
    print("已在项目根目录，跳过克隆")

!which python3 && ln -sf $(which python3) /usr/local/bin/python 2>/dev/null; python --version && echo "python ready"

PROJECT = os.getcwd()
print(f"\n工作目录: {PROJECT}")
!ls -F | head -30

# ===== 2. 安装依赖 =====
print("\n===== 安装 PyTorch + 依赖 =====")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q -r requirements.txt
print("依赖安装完成！")

# ===== 3. 检查目标字体（智能全工作区搜索）=====
print("\n===== 检查字体 =====")

def find_font_in_all_roots(filename, roots=None):
    if roots is None:
        roots = list(set(WORKSPACE_ROOTS + [os.getcwd(), PROJECT]))
    seen_paths = set()
    for search_root in roots:
        real_root = os.path.realpath(search_root)
        if real_root in seen_paths or not os.path.isdir(search_root):
            continue
        seen_paths.add(real_root)
        for root, dirs, files in os.walk(search_root, followlinks=False):
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in ("__pycache__", "node_modules")]
            for f in files:
                if f.lower() == filename.lower():
                    return os.path.join(root, f)
    return None

font_path = f"fonts/{TARGET_FONT}"

print("当前 fonts/ 目录内容:")
if os.path.isdir("fonts"):
    !find fonts/ -type f 2>/dev/null | head -30
else:
    print("  fonts/ 不存在（将在找到字体后自动创建）")

if not os.path.exists(font_path):
    print(f"\n[INFO] {font_path} 不存在，正在搜索整个工作区...")
    found = find_font_in_all_roots(TARGET_FONT)
    if found:
        print(f"[OK] 找到字体: {found}")
        os.makedirs("fonts", exist_ok=True)
        shutil.copy2(found, font_path)
        print(f"[OK] 已复制到: {font_path}")
    else:
        print("\n[ERROR] 字体未找到！列出工作区中的 .otf/.ttf 文件:")
        for r in WORKSPACE_ROOTS:
            for ext in ["*.otf", "*.ttf"]:
                matches = glob.glob(os.path.join(r, "**", ext), recursive=True)
                user_fonts = [m for m in matches if "/usr/" not in m and "jigmo" not in m.lower()]
                if user_fonts:
                    print(f"  [{r}]:")
                    for mf in user_fonts[:20]:
                        print(f"    - {mf}")
        raise FileNotFoundError(
            f"\n字体文件 '{TARGET_FONT}' 在工作区都找不到！\n"
            f"请检查:\n"
            f"  1. 文件名是否完全一致（注意大小写）? 当前配置: '{TARGET_FONT}'\n"
            f"  2. 字体是否已上传到 ModelScope Workspace?\n"
            f"  3. 上方是否列出了其他 .otf/.ttf 文件？如果有，请将 Cell 0 中 TARGET_FONT 改为对应文件名"
        )
else:
    print(f"目标字体已就绪: {font_path}")

# ===== 4. 把字体路径写进所有 .sh 脚本 =====
print("\n===== 改写脚本字体路径 =====")
for sh_file in glob.glob("scripts/*.sh"):
    with open(sh_file, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r'fonts/[\w.-]+\.(ttf|otf)', f'fonts/{TARGET_FONT}', content)
    with open(sh_file, "w", encoding="utf-8") as f:
        f.write(content)
print("脚本字体路径已统一替换")

# ===== 5. Jigmo 参考字体：本地搜索优先 =====
print("\n===== 准备 Jigmo 参考字体 =====")

jigmo_files = ["jigmo.ttf", "jigmo2.ttf", "jigmo3.ttf"]
jigmo_missing = []

def validate_font_file(fpath):
    try:
        f = TTFont(fpath)
        if "cmap" not in f: return False, "缺少 cmap 表"
        return True, f"OK ({len(f.getBestCmap())} glyphs)"
    except Exception as e:
        return False, str(e)[:80]

os.makedirs("fonts/jigmo", exist_ok=True)
for fname in jigmo_files:
    fpath = f"fonts/jigmo/{fname}"
    if os.path.exists(fpath):
        valid, msg = validate_font_file(fpath)
        if valid:
            print(f"  [OK] {fname}: {msg}")
        else:
            print(f"  [WARN] {fname} 无效 ({msg})，将重新搜索")
            os.remove(fpath)
            jigmo_missing.append(fname)
    else:
        jigmo_missing.append(fname)

if jigmo_missing:
    print(f"\n  需要 {len(jigmo_missing)} 个 Jigmo 文件: {jigmo_missing}")
    print("  正在本地搜索（递归扫描工作区）...")
    still_missing = []
    for fname in jigmo_missing:
        found = find_font_in_all_roots(fname)
        if found:
            target = f"fonts/jigmo/{fname}"
            shutil.copy2(found, target)
            valid, msg = validate_font_file(target)
            if valid:
                print(f"  [OK] {fname}: 从 {found} 复制，{msg}")
            else:
                print(f"  [WARN] {fname}: 验证失败 ({msg})")
                still_missing.append(fname)
        else:
            still_missing.append(fname)
    
    if still_missing:
        print(f"\n  仍未找到单个字体文件，尝试搜索 Jigmo ZIP...")
        found_zip = None
        for search_root in WORKSPACE_ROOTS + [PROJECT]:
            if not os.path.isdir(search_root): continue
            for root, dirs, files in os.walk(search_root, followlinks=False):
                for f in files:
                    if "jigmo" in f.lower() and f.lower().endswith(".zip"):
                        found_zip = os.path.join(root, f)
                        break
                if found_zip: break
            if found_zip: break
        
        if found_zip:
            print(f"  找到 Jigmo ZIP: {found_zip}")
            zf = zipfile.ZipFile(found_zip)
            for zn in zf.namelist():
                bn = os.path.basename(zn).lower()
                if bn in jigmo_files:
                    zf.extract(zn, "fonts/jigmo")
                    extracted = os.path.join("fonts/jigmo", zn)
                    target = os.path.join("fonts/jigmo", bn)
                    if extracted != target:
                        if os.path.exists(target): os.remove(target)
                        os.rename(extracted, target)
                    if bn in still_missing: still_missing.remove(bn)
            zf.close()
            print(f"  [OK] 从 ZIP 提取了 Jigmo 字体")

        if still_missing:
            print(f"\n  [WARN] 仍缺少的 Jigmo: {still_missing}")
            print("  尝试从网络下载...（如果失败则中止）")
            def download_jigmo_fonts():
                zip_url = "https://kamichikoichi.github.io/jigmo/Jigmo-20250912.zip"
                resp = urllib.request.urlopen(zip_url, timeout=60)
                data = resp.read()
                if len(data) < 10000: raise ValueError(f"太小 ({len(data)} bytes)")
                zf = zipfile.ZipFile(io.BytesIO(data))
                for zn in zf.namelist():
                    bn = os.path.basename(zn).lower()
                    if bn in jigmo_files:
                        zf.extract(zn, "fonts/jigmo")
                        extracted = os.path.join("fonts/jigmo", zn)
                        target = os.path.join("fonts/jigmo", bn)
                        if extracted != target:
                            if os.path.exists(target): os.remove(target)
                            os.rename(extracted, target)
                zf.close()
                return True
            try:
                download_jigmo_fonts()
                print("  [OK] Jigmo 在线下载成功")
                still_missing = []
            except Exception as e:
                print(f"  [ERROR] 在线下载失败: {e}")
        
        if still_missing:
            print("\n  工作区中现有的 Jigmo 相关文件:")
            jigmo_related = []
            for sr in WORKSPACE_ROOTS + [PROJECT]:
                if not os.path.isdir(sr): continue
                for root, dirs, files in os.walk(sr, followlinks=False):
                    for f in files:
                        if "jigmo" in f.lower():
                            jigmo_related.append(os.path.join(root, f))
            for jr in jigmo_related[:20]:
                size = os.path.getsize(jr) / 1024 if os.path.isfile(jr) else 0
                print(f"    - {jr} ({size:.0f} KB)")
            raise RuntimeError(
                f"\n缺少 Jigmo 参考字体: {still_missing}\n"
                f"请在 ModelScope Workspace 上传 jigmo.ttf/jigmo2.ttf/jigmo3.ttf，\n"
                f"或上传包含它们的 Jigmo ZIP 包，然后重跑本 Cell。\n"
                f"来源: https://kamichikoichi.github.io/jigmo/"
            )
    print("\n  最终验证:")
    for fname in jigmo_files:
        valid, msg = validate_font_file(f"fonts/jigmo/{fname}")
        status_str = "OK" if valid else "FAIL"
        print(f"    [{status_str}] {fname}: {msg}")
        if not valid: raise RuntimeError(f"Jigmo 字体 {fname} 验证失败！")
else:
    print("Jigmo 字体已全部就绪")

# ===== 6. 断连自检 + 字体切换检测 + 自动续训改写 =====
print("\n===== 断连自检 + 字体切换检测 =====")
os.makedirs("checkpoints", exist_ok=True)

def load_state():
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r", encoding="utf-8") as f: return json.load(f)
        except: pass
    return {}

def save_state(state):
    with open(STATE_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

state = load_state()
prev_data_font = state.get("data_font")
if prev_data_font and prev_data_font != FONT_NAME:
    print(f"  [字体切换] {prev_data_font} -> {FONT_NAME}")
    print("    Cell 2 将重新执行数据准备")
    state["data_prep_done"] = False
    state["data_font"] = None

state.setdefault("font", FONT_NAME)
save_state(state)

vqvae_ckpt = f"checkpoints/vqvae_{FONT_NAME}.pth"
ldm_ckpt = f"checkpoints/ldm_{FONT_NAME}.pth"
have_vqvae = os.path.exists(vqvae_ckpt)
have_ldm = os.path.exists(ldm_ckpt)

print(f"  数据准备: {'已完成' if state.get('data_prep_done') else '未完成'}（绑定字体: {state.get('data_font') or '无'}）")
print(f"  VQ-VAE 检查点: {'存在' if have_vqvae else '不存在'}")
print(f"  LDM 检查点:    {'存在' if have_ldm else '不存在'}")

def _set_resume_from(sh_path, ckpt):
    with open(sh_path, "r", encoding="utf-8") as f: content = f.read()
    content = re.sub(r'RESUME_FROM="[^"]*"', f'RESUME_FROM="{ckpt}"', content)
    with open(sh_path, "w", encoding="utf-8") as f: f.write(content)

for sh_file in sorted(glob.glob("scripts/train_vqvae*.sh")):
    _set_resume_from(sh_file, vqvae_ckpt if have_vqvae else "")
for sh_file in sorted(glob.glob("scripts/train_ldm*.sh")):
    _set_resume_from(sh_file, ldm_ckpt if have_ldm else "")

resume_label = vqvae_ckpt if have_vqvae else "(空，从零训练)"
print(f"  train_vqvae RESUME_FROM -> {resume_label}")

# ===== 7. GPU 软检测 + 按实例自动调优 =====
print("\n===== 硬件检测与性能调优 =====")
import torch

cpu_cores = os.cpu_count() or 4

def _avail_mem_gb():
    try:
        with open("/sys/fs/cgroup/memory.max", "r") as f:
            v = f.read().strip()
            if v.isdigit() and int(v) > 0: return int(v) / 1024 ** 3
    except: pass
    try:
        with open("/proc/meminfo", "r") as f:
            for line in f:
                if line.startswith("MemTotal"): return float(line.split()[1]) / 1024 ** 2
    except: pass
    return 8.0

avail_mem_gb = _avail_mem_gb()
DL_WORKERS = max(2, min(8, cpu_cores))
RENDER_WORKERS = max(2, min(16, cpu_cores))
if avail_mem_gb < 4: RENDER_WORKERS = DL_WORKERS = 2
elif avail_mem_gb < 8: RENDER_WORKERS = min(4, RENDER_WORKERS); DL_WORKERS = min(2, DL_WORKERS)
print(f"  可用内存: {avail_mem_gb:.1f} GB")

gpu_name, vram_gb = None, 0.0
if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = prop.total_memory / 1024**3
    print(f"GPU: {gpu_name} ({vram_gb:.1f} GB) | CUDA {torch.version.cuda}")
else:
    print("未检测到 GPU：当前实例为 CPU 规格")
    print("  -> 可运行: Cell 2（数据准备）/ Cell 6-8（SVG转换）")
    print("  -> 不可运行: Cell 3/4（训练）/ Cell 5（推理），请切换 GPU 实例")

profile = HW_PROFILE
if profile == "auto":
    if gpu_name is None: profile = "cpu"
    elif any(k in gpu_name.upper() for k in ("V100","L40","L20","A100","H100")): profile = "v100"
    elif any(k in gpu_name.upper() for k in ("A10","A16","L4","L2")): profile = "a10"
    elif "T4" in gpu_name.upper(): profile = "t4"
    elif vram_gb >= 28: profile = "v100"
    elif vram_gb >= 20: profile = "a10"
    else: profile = "t4"

if profile == "a10": DL_WORKERS = min(16, max(8, cpu_cores - 6))
elif profile in ("v100","t4"): DL_WORKERS = max(2, min(6, cpu_cores - 2))
else: DL_WORKERS = max(2, min(8, cpu_cores))

if profile == "v100": VQVAE_BS, LDM_BS, EVAL_BS, INFER_BS = 88, 128, 16, 32
elif profile == "a10": VQVAE_BS, LDM_BS, EVAL_BS, INFER_BS = 64, 128, 16, 32
elif profile == "t4": VQVAE_BS, LDM_BS, EVAL_BS, INFER_BS = 44, 64, 8, 16
else: VQVAE_BS = LDM_BS = EVAL_BS = INFER_BS = None

print(f"\n性能档位: {HW_PROFILE} -> {profile} | CPU 核数: {cpu_cores}")
print(f"  DataLoader workers = {DL_WORKERS}")
print(f"  字形渲染 workers   = {RENDER_WORKERS}")
if profile != "cpu":
    print(f"  VQ-VAE batch = {VQVAE_BS} | LDM batch = {LDM_BS} | Eval batch = {EVAL_BS} | Inference batch = {INFER_BS}")

def set_sh_var(sh_path, var, value):
    with open(sh_path, encoding="utf-8") as f: lines = f.readlines()
    for i, ln in enumerate(lines):
        if ln.startswith(var + "="):
            body = ln[len(var)+1:].rstrip("\n")
            tail = ""
            if "#" in body: tail = "  " + body[body.index("#"):]
            lines[i] = f"{var}={value}{tail}\n"
            break
    with open(sh_path, "w", encoding="utf-8") as f: f.writelines(lines)

set_sh_var("scripts/prepare_dataset.sh", "NUM_WORKERS", RENDER_WORKERS)
set_sh_var("scripts/extract_charset.sh", "DEVICE", '"cuda"' if gpu_name else '"cpu"')

VQVAE_TRAIN_SCRIPT = None
LDM_TRAIN_SCRIPT = None
if profile != "cpu":
    VQVAE_TRAIN_SCRIPT = f"scripts/train_vqvae_{profile.upper()}.sh"
    LDM_TRAIN_SCRIPT = f"scripts/train_ldm_{profile.upper()}.sh"
    if not os.path.exists(VQVAE_TRAIN_SCRIPT):
        VQVAE_TRAIN_SCRIPT = "scripts/train_vqvae.sh"
        set_sh_var(VQVAE_TRAIN_SCRIPT, "BATCH_SIZE", VQVAE_BS)
        set_sh_var(VQVAE_TRAIN_SCRIPT, "NUM_WORKERS", DL_WORKERS)
    if not os.path.exists(LDM_TRAIN_SCRIPT):
        LDM_TRAIN_SCRIPT = "scripts/train_ldm.sh"
        set_sh_var(LDM_TRAIN_SCRIPT, "BATCH_SIZE", LDM_BS)
        set_sh_var(LDM_TRAIN_SCRIPT, "NUM_WORKERS", DL_WORKERS)
        set_sh_var(LDM_TRAIN_SCRIPT, "EVAL_BATCH_SIZE", EVAL_BS)
    set_sh_var("scripts/inference.sh", "BATCH_SIZE", INFER_BS)
    set_sh_var("scripts/compute_metrics.sh", "EVAL_BATCH_SIZE", EVAL_BS)
    print(f"\n[训练脚本] VQ-VAE -> {VQVAE_TRAIN_SCRIPT}")
    print(f"[训练脚本] LDM     -> {LDM_TRAIN_SCRIPT}")

print(f"\n全部初始化完成！字体: {font_path}")

---
## Cell 2: 数据准备（约 10-20 分钟，断连可恢复）

分析字体覆盖率 -> 渲染字形图片 -> 提取训练/验证字符集
建议使用 CPU 实例。已完成且字体未变更会自动跳过。

In [ ]:

import os, json, subprocess, sys

if not DO_DATA_PREP:
    print("DO_DATA_PREP=False，跳过 Cell 2")
else:
    def _load_state():
        if os.path.exists(STATE_FILE):
            try:
                with open(STATE_FILE, "r", encoding="utf-8") as f: return json.load(f)
            except: pass
        return {}
    def _save_state(s):
        with open(STATE_FILE, "w", encoding="utf-8") as f:
            json.dump(s, f, ensure_ascii=False, indent=2)

    def _run_script(script_path, step_name):
        print(f"\n{'='*60}")
        print(f"  执行: {step_name}")
        print(f"  脚本: {script_path}")
        print(f"{'='*60}")
        result = subprocess.run(["bash", script_path], capture_output=True, text=True)
        if result.stdout: print(result.stdout)
        if result.returncode != 0:
            if result.stderr:
                print("[STDERR]", file=sys.stderr)
                print(result.stderr, file=sys.stderr)
            raise RuntimeError(f"{step_name} 失败！返回码: {result.returncode}")
        print(f"[OK] {step_name} 完成")

    state = _load_state()
    data_done   = os.path.isdir("data/reference") and os.path.isdir("data/target")
    splits_done = os.path.exists(f"charsets/splits/{FONT_NAME}/train.txt") and \
                  os.path.exists(f"charsets/splits/{FONT_NAME}/val.txt")
    same_font   = state.get("data_font") == FONT_NAME

    if state.get("data_prep_done") and same_font and data_done and splits_done:
        print(f"字体 {FONT_NAME} 的数据准备已完成，跳过 Cell 2")
    else:
        if not same_font and state.get("data_font"):
            print(f"[字体切换] {state['data_font']} -> {FONT_NAME}")
        _run_script("scripts/analyze_font.sh",    "Step 1/3: 分析字体覆盖率")
        _run_script("scripts/prepare_dataset.sh", "Step 2/3: 生成数据集图片")
        _run_script("scripts/extract_charset.sh", "Step 3/3: 提取训练/验证字符集")

        if not (os.path.isdir("data/reference") and os.path.isdir("data/target")):
            raise RuntimeError("data/ 目录未正确生成")
        if not os.path.exists(f"charsets/splits/{FONT_NAME}/train.txt"):
            raise RuntimeError("train.txt 未生成")

        state["data_prep_done"] = True
        state["data_font"] = FONT_NAME
        _save_state(state)
        print("\n===== 数据准备完成 =====")


---
## Cell 3: 训练 VQ-VAE（约 6-8 小时，自动精确续训）

切换到 GPU 实例，重跑 Cell 0 + Cell 1 后再运行本 Cell。

In [ ]:

import os, subprocess, torch

if not DO_TRAIN_VQVAE:
    print("DO_TRAIN_VQVAE=False，跳过 Cell 3")
else:
    assert torch.cuda.is_available(), "Cell 3 需要 GPU 实例。切换到 GPU 后重跑 Cell 0 + Cell 1，再运行本 Cell"
    assert os.path.isdir("data/target"), "data/ 不存在，请先运行 Cell 2 完成数据准备"
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    assert VQVAE_TRAIN_SCRIPT is not None, "请先运行 Cell 1 完成硬件检测与训练脚本选择"
    print(f"将执行训练脚本: {VQVAE_TRAIN_SCRIPT}")
    subprocess.run(["bash", VQVAE_TRAIN_SCRIPT], check=True)


---
## Cell 4前置操作：离线放置 VGG16 权重

手动把 `vgg16-397923af.pth` 上传到 ModelScope Workspace，本 Cell 会自动搜索并移动到缓存目录。

本地文件下载备份：GitHub Raw URL (见原始 Colab notebook)

In [ ]:

import os, shutil

search_r = WORKSPACE_ROOTS + ["/root", os.getcwd(), os.path.dirname(os.getcwd())] if 'WORKSPACE_ROOTS' in dir() else ["/workspace", "/home", "/root", os.getcwd()]

def find_file(filename, roots):
    for root in roots:
        if os.path.isdir(root):
            for dp, dn, fn in os.walk(root):
                if filename in fn: return os.path.join(dp, filename)
    return None

src = find_file("vgg16-397923af.pth", search_r)
if src is None:
    print("没找到 vgg16-397923af.pth，请确认文件已上传到工作空间")
    print(f"当前工作目录: {os.getcwd()}")
else:
    dst_dir = "/root/.cache/torch/hub/checkpoints"
    dst = os.path.join(dst_dir, "vgg16-397923af.pth")
    os.makedirs(dst_dir, exist_ok=True)
    if os.path.abspath(src) != os.path.abspath(dst):
        shutil.move(src, dst)
    print(f"VGG16 权重已就位: {dst}")


---
## Cell 4: 训练 LDM（约 10-15 小时，自动精确续训）

依赖 vqvae checkpoint。断连恢复：重跑 Cell 0、Cell 1，再运行本 Cell。

In [ ]:

import os, subprocess, torch

vqvae_ckpt = f"checkpoints/vqvae_{FONT_NAME}.pth"

if not DO_TRAIN_LDM:
    print("DO_TRAIN_LDM=False，跳过 Cell 4")
else:
    assert torch.cuda.is_available(), "Cell 4 需要 GPU 实例"
    assert os.path.exists(vqvae_ckpt), f"{vqvae_ckpt} 不存在，请先完成 Cell 3 训练 VQ-VAE"
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    assert LDM_TRAIN_SCRIPT is not None, "请先运行 Cell 1 完成硬件检测与训练脚本选择"
    print(f"将执行训练脚本: {LDM_TRAIN_SCRIPT}")
    subprocess.run(["bash", LDM_TRAIN_SCRIPT], check=True)


---
## Cell 5: 推理生成 + 评估指标

补字基准在 Cell 0 的 CHARSET_BASE 选择。T4 GPU 即可。

In [ ]:

import os, re, subprocess, torch

ldm_ckpt = f"checkpoints/ldm_{FONT_NAME}.pth"

if not DO_INFERENCE:
    print("DO_INFERENCE=False，跳过 Cell 5")
else:
    assert torch.cuda.is_available(), "Cell 5 需要 GPU 实例"
    assert os.path.exists(ldm_ckpt), f"{ldm_ckpt} 不存在，请先完成 Cell 4 训练 LDM"

    def build_charset_path(base):
        if base in ("gbk", "gb2312"):
            from fontTools.ttLib import TTFont
            base_chars = set()
            for cp in range(0x4E00, 0xA000):
                try: chr(cp).encode(base); base_chars.add(chr(cp))
                except UnicodeEncodeError: pass
            font = TTFont(f"fonts/{TARGET_FONT}", fontNumber=0)
            cmap = set()
            for table in font["cmap"].tables:
                if table.isUnicode(): cmap.update(table.cmap.keys())
            missing = sorted(base_chars - {chr(c) for c in cmap})
            out_dir = f"charsets/{base}_coverage/{FONT_NAME}"
            os.makedirs(out_dir, exist_ok=True)
            out_path = f"{out_dir}/missing.txt"
            with open(out_path, "w", encoding="utf-8") as f: f.write("\n".join(missing))
            print(f"[{base}] 基准 {len(base_chars)} 字，缺失 {len(missing)} 字 -> {out_path}")
            return out_path
        if base == "unihan":
            p = f"charsets/unihan_coverage/{FONT_NAME}/missing.txt"
        else:
            p = f"charsets/jf7000_coverage/{FONT_NAME}/missing.txt"
        if not os.path.exists(p):
            raise FileNotFoundError(f"{p} 不存在，请先运行 Cell 2")
        return p

    charset_path = build_charset_path(CHARSET_BASE)
    print(f"补字基准: {CHARSET_BASE} -> {charset_path}")

    with open("scripts/inference.sh", encoding="utf-8") as f: content = f.read()
    content = re.sub(r'CHARSET_PATH="[^"]*"', f'CHARSET_PATH="{charset_path}"', content)
    content = re.sub(r'DEVICE="[^"]*"', 'DEVICE="cuda"', content)
    with open("scripts/inference.sh", "w", encoding="utf-8") as f: f.write(content)
    print("inference.sh 已指向补字基准字符集")

    print("\n===== 推理生成 =====")
    subprocess.run(["bash", "scripts/inference.sh"], check=True)
    print("\n===== 计算评估指标 =====")
    subprocess.run(["bash", "scripts/compute_metrics.sh"], check=True)

    print(f"\n===== GPU 阶段全部完成 =====")
    print(f"生成 PNG 位置: samples_{FONT_NAME}/inference/gen/")
    print("下一步：SVG 向量化为纯 CPU 任务，切换到 CPU 实例运行 Cell 6-8")


---
## Cell 6: 转换 SVG + 产出总览

纯 CPU 任务，建议多核实例。

In [ ]:

import os, subprocess

png_dir = f"samples_{FONT_NAME}/inference/gen"
svg_dir = f"svgs_{FONT_NAME}"

if not os.path.isdir(png_dir):
    print(f"{png_dir} 不存在，请先在 GPU 实例完成 Cell 5 推理")
else:
    print("本 Cell 为纯 CPU 任务，无需 GPU")
    print("\n===== 转换 SVG（1万+ 文件约需 10-30 分钟）=====")
    subprocess.run(["bash", "scripts/convert_to_svg.sh"], check=True)

    print("\n===== 产出总览 =====")
    n_png = len([f for f in os.listdir(png_dir) if f.endswith(".png")])
    n_svg = len([f for f in os.listdir(svg_dir) if f.endswith(".svg")]) if os.path.isdir(svg_dir) else 0
    print(f"生成 PNG: {n_png} 张 -> {png_dir}/")
    print(f"转换 SVG: {n_svg} 个 -> {svg_dir}/")

    print("\n[checkpoints]")
    if os.path.isdir("checkpoints"):
        for f in sorted(os.listdir("checkpoints")):
            size = os.path.getsize(os.path.join("checkpoints", f)) / 1024**2
            print(f"  {f}  ({size:.1f} MB)")
    else:
        print("  无")

    print(f"\n[samples_{FONT_NAME}]")
    if os.path.isdir(f"samples_{FONT_NAME}"):
        for d in sorted(os.listdir(f"samples_{FONT_NAME}")):
            print(f"  {d}/")


---
## Cell 7: 导出并下载 SVG 结果

In [ ]:

import os, glob, zipfile, io, base64
from IPython.display import HTML, display

svg_dir = f"svgs_{FONT_NAME}"
if not os.path.isdir(svg_dir):
    print(f"{svg_dir} 不存在，请先运行 Cell 6 完成转换")
else:
    svg_files = sorted(glob.glob(os.path.join(svg_dir, "*.svg")))
    print(f"共找到 {len(svg_files)} 个 SVG 文件")

    zip_buffer = io.BytesIO()
    with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in svg_files:
            zf.write(f, arcname=os.path.basename(f))
    zip_data = zip_buffer.getvalue()
    zip_name = f"svgs_{FONT_NAME}.zip"

    saved_paths = []
    roots = [os.path.abspath(os.getcwd()), os.path.dirname(os.path.abspath(os.getcwd())),
             "/workspace", "/home", "/root", "/mnt/workspace"]
    seen = set()
    for r in roots:
        if r and r not in seen and os.path.isdir(r):
            seen.add(r)
            try:
                p = os.path.join(r, zip_name)
                with open(p, "wb") as f: f.write(zip_data)
                saved_paths.append(p)
            except: pass
    for p in saved_paths:
        print(f"  [已保存] {p} ({len(zip_data)/1024:.0f} KB)")

    b64 = base64.b64encode(zip_data).decode()
    download_link = (
        '<a href="data:application/zip;base64,' + b64 + f'" download="{zip_name}" '
        'style="display:inline-block;font-size:18px;font-weight:bold;color:#fff;'
        'background:#1a73e8;padding:12px 28px;border-radius:8px;text-decoration:none;">'
        f"下载全部 SVG ({len(svg_files)} 个 / {len(zip_data)/1024:.0f} KB)</a>"
    )
    display(HTML(download_link))

    cards = []
    for f in svg_files[:9]:
        with open(f, encoding="utf-8") as fh: svg = fh.read()
        svg = svg.replace("<svg ", '<svg width="80" height="80" style="background:#fff;" ', 1)
        name = os.path.splitext(os.path.basename(f))[0]
        cards.append(
            f'<div style="border:1px solid #e0e0e0;border-radius:8px;padding:8px;'
            f'text-align:center;width:96px;">{svg}<div style="font-size:12px;color:#555;">{name}</div></div>'
        )
    display(HTML('<div style="display:flex;flex-wrap:wrap;gap:10px;">' + "".join(cards) + "</div>"))


---
## Cell 8: 导出全部生成结果（PNG + SVG）为 zip

In [ ]:

import os, glob, zipfile, io, base64
from IPython.display import HTML, display

FONT = FONT_NAME
include_dirs = [
    f"svgs_{FONT}",
    f"samples_{FONT}/inference/gen",
    f"samples_{FONT}/inference/gt",
    f"samples_{FONT}/inference/ref",
    f"samples_{FONT}/val",
    f"samples_{FONT}/train",
]

zip_name = f"hanzigen_output_{FONT}.zip"
buf = io.BytesIO()
total_files = 0
found_any = False

with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in include_dirs:
        if not os.path.isdir(folder):
            print(f"  [跳过] {folder} 不存在")
            continue
        found_any = True
        files = [p for p in glob.glob(os.path.join(folder, "**", "*.*"), recursive=True) if os.path.isfile(p)]
        for f in files:
            zf.write(f, arcname=os.path.relpath(f))
        print(f"  [OK] {folder} -> {len(files)} 个文件")
        total_files += len(files)

if not found_any:
    print("所有目录都不存在，请先运行 Cell 5 / Cell 6 生成结果")
else:
    zip_data = buf.getvalue()
    saved_paths = []
    roots = [os.path.abspath(os.getcwd()), os.path.dirname(os.path.abspath(os.getcwd())),
             "/workspace", "/home", "/root", "/mnt/workspace"]
    seen = set()
    for r in roots:
        if r and r not in seen and os.path.isdir(r):
            seen.add(r)
            try:
                p = os.path.join(r, zip_name)
                with open(p, "wb") as f: f.write(zip_data)
                saved_paths.append(p)
            except: pass
    print(f"\n共 {total_files} 个文件，{len(zip_data)/1024/1024:.1f} MB")
    for p in saved_paths:
        print(f"  [已保存] {p}")

    b64 = base64.b64encode(zip_data).decode()
    download_link = (
        '<a href="data:application/zip;base64,' + b64 + f'" download="{zip_name}" '
        'style="display:inline-block;font-size:18px;font-weight:bold;color:#fff;'
        'background:#188038;padding:12px 28px;border-radius:8px;text-decoration:none;">'
        f"下载全部结果 ({total_files} 个文件 / {len(zip_data)/1024/1024:.1f} MB)</a>"
    )
    display(HTML(download_link))


---
## Cell CK1: Checkpoint 导出（跨平台迁移·源端）

在中止训练的平台上运行，把 checkpoint + 字体 + charsets 打包带走。
新平台上运行 Cell CK2 导入即可。
不需要搬运庞大的 data/ 图像数据集。

In [ ]:
import os, shutil, zipfile
from pathlib import Path

start_dir = Path.cwd()  # notebook 原始工作目录（持久化目录，zip 输出到这里便于下载）

# ============ 自动定位项目根目录 ============
# 与 Cell 1 相同：确保从真正的项目根目录打包，而非 notebook 工作目录。
def _find_project_root():
    if os.path.isdir("scripts") and os.path.exists("requirements.txt"):
        return Path.cwd()
    for search_root in [os.getcwd(), "/mnt/workspace", "/workspace", "/home", "/root", "/"]:
        if not os.path.isdir(search_root):
            continue
        for entry in sorted(os.listdir(search_root)):
            cand = os.path.join(search_root, entry)
            if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "scripts")):
                return Path(cand)
    return None

_proj = _find_project_root()
if _proj is None:
    raise SystemExit("未找到项目根目录（含 scripts/），请先运行 Cell 1。")
os.chdir(_proj)
print(f"已切换到项目根目录: {os.getcwd()}")
# ===========================================

DIRS_TO_PACK = ["checkpoints", "fonts", "charsets"]
OUTPUT_ZIP = start_dir / "hanzi_transfer.zip"

missing = [d for d in DIRS_TO_PACK if not Path(d).exists()]
if missing: print("以下目录不存在将被跳过:", missing)
present = [d for d in DIRS_TO_PACK if Path(d).exists()]

ckpt_files = list(Path("checkpoints").glob("*.pth")) if Path("checkpoints").exists() else []
if "checkpoints" in present and not ckpt_files:
    print("checkpoints/ 没有 .pth 文件！")
if not present:
    raise SystemExit("没有任何可打包目录")

if Path(OUTPUT_ZIP).exists(): Path(OUTPUT_ZIP).unlink()

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for d in present:
        for root, _, files in os.walk(d):
            for f in files:
                fp = Path(root) / f
                zf.write(fp, fp)

size_mb = Path(OUTPUT_ZIP).stat().st_size / 1024 / 1024
print(f"已导出: {Path(OUTPUT_ZIP).resolve()}  ({size_mb:.1f} MB)")
print("包含目录:", present)
print(f"checkpoint 数量: {len(ckpt_files)}")

parent_dir = Path(OUTPUT_ZIP).parent.parent
alt_path = parent_dir / "hanzi_transfer.zip"
if alt_path != OUTPUT_ZIP and parent_dir.exists():
    try:
        shutil.copy2(OUTPUT_ZIP, alt_path)
        print(f"另存: {alt_path.resolve()}")
    except: pass

print("\n将此 zip 下载到本地，上传到新平台后运行 Cell CK2。")

---
## Cell CK2: Checkpoint 导入（跨平台迁移·目标端）

把从 Cell CK1 导出的 hanzi_transfer.zip 上传到本平台，运行本 Cell 解压。
解压后仍需跑 Cell 2 重建 data/，再 resume 训练。

In [ ]:
import os, zipfile
from pathlib import Path

start_dir = Path.cwd()  # notebook 原始工作目录（zip 通常上传到这里）

# ============ 自动定位项目根目录 ============
# 与 Cell 1 相同：确保解压到真正的项目根目录，而非 notebook 工作目录。
def _find_project_root():
    if os.path.isdir("scripts") and os.path.exists("requirements.txt"):
        return Path.cwd()
    for search_root in [os.getcwd(), "/mnt/workspace", "/workspace", "/home", "/root", "/"]:
        if not os.path.isdir(search_root):
            continue
        for entry in sorted(os.listdir(search_root)):
            cand = os.path.join(search_root, entry)
            if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "scripts")):
                return Path(cand)
    return None

_proj = _find_project_root()
if _proj is None:
    raise SystemExit("未找到项目根目录（含 scripts/），请先运行 Cell 1 获取项目代码。")
os.chdir(_proj)
print(f"已切换到项目根目录: {os.getcwd()}")
# ===========================================

ZIP_FILENAME = "hanzi_transfer.zip"

# 搜索 zip：优先 start_dir（notebook 原始工作目录），再全工作区
zip_candidates = []
search_roots = [str(start_dir)]
if 'WORKSPACE_ROOTS' in dir():
    search_roots = list(WORKSPACE_ROOTS) + search_roots
search_roots += [os.getcwd(), "/mnt/workspace", "/workspace", "/home", "/root"]
seen = set()
for s in search_roots:
    if not os.path.isdir(s): continue
    real = os.path.realpath(s)
    if real in seen: continue
    seen.add(real)
    for root, dirs, files in os.walk(s, followlinks=False):
        if ZIP_FILENAME in files:
            zip_candidates.append(os.path.join(root, ZIP_FILENAME))

if zip_candidates:
    zip_path = zip_candidates[0]
    print(f"找到 hanzi_transfer.zip: {zip_path}")
else:
    print(f"找不到 hanzi_transfer.zip！搜索范围: {search_roots}")
    raise SystemExit("Zip 文件未找到。")

EXTRACT_ROOT = "."   # 已 chdir 到项目根目录，解压到此处
with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()
    zf.extractall(EXTRACT_ROOT)

print(f"已解压 {len(names)} 个文件到 {os.getcwd()}")
top_dirs = sorted({n.split('/')[0] for n in names if '/' in n})
print("还原的顶层目录:", top_dirs)

ckpt = list(Path("checkpoints").glob("*.pth")) if Path("checkpoints").exists() else []
print(f"\ncheckpoints/*.pth 数量: {len(ckpt)}")
for p in ckpt: print(" -", p.name)

if ckpt:
    all_names = [p.stem for p in ckpt]
    fonts_found = set()
    for an in all_names:
        if an.startswith("vqvae_"): fonts_found.add(an[6:])
        elif an.startswith("ldm_"): fonts_found.add(an[4:])
    if len(fonts_found) == 1:
        inferred_font = fonts_found.pop()
        print(f"\n检测到字体: {inferred_font}")
        print("请将 Cell 0 中 TARGET_FONT 设为对应的字体文件名，然后重跑 Cell 1")

print("\n下一步：按原流程跑 Cell 2（数据准备）重建 data/，再运行训练 Cell；")
print("Cell 1 会自动检测到 checkpoints 并 resume 继续训练。")

---
## 常见问题

| 问题 | 解决 |
|------|------|
| 找不到字体文件 | Cell 1 会扫描整个工作区，确认文件名和 Cell 0 配置一致（区分大小写）|
| 找不到 Jigmo | 手动上传 jigmo.ttf/jigmo2.ttf/jigmo3.ttf 到 ModelScope Workspace，重跑 Cell 1 |
| 找不到 hanzi_transfer.zip | 确认文件已上传，Cell CK2 会在整个工作区搜索 |
| CPU 实例跑 Cell 1 报 GPU 错误 | 新版为软检测，不会报错；Cell 3/4/5 才有硬断言 |
| 训练 OOM | Cell 0 把 HW_PROFILE 改为 "t4" 重跑 Cell 1 |
| 换新字体训练 | 直接 Cell 0 改名即可 |
| 只想补 GBK 简体缺字 | Cell 0 设 CHARSET_BASE="gbk" |
| SVG 阶段内存不足崩溃 | 换大内存 CPU 实例 |